# POPE × Phi-4-multimodal-instruct — Inference trên tập con 900 mẫu
Nhóm 3 — thành viên phụ trách `microsoft/Phi-4-multimodal-instruct`. Notebook này **giữ nguyên quy trình, prompt, tham số sinh và định dạng output của notebook Qwen3** (`benchmarks/pope/Qwen3/`) để ba model so sánh được; chỉ thay phần nạp model/ảnh cho Phi-4 và dùng **tập con cố định 900 mẫu** (300 mỗi category `adversarial` / `popular` / `random`, seed 42).

**Cách chạy trên Kaggle**
1. *Settings*: Accelerator **GPU T4 ×2**, Internet **On**, Persistence nên bật.
2. Chạy từ cell 1.1. Cell 1.2 sẽ cài `transformers==4.48.2`; nếu nó dừng với thông báo cần restart → **Run → Restart Session**, rồi *Run All* lại.
3. Cell 5.2 chạy thử 5 mẫu và in ước lượng thời gian. Muốn thử nhanh hơn: đặt `MAX_SAMPLES = 30` ở cell 1.3 (nhớ đổi lại `None` khi chạy thật; các mẫu đã xong sẽ được giữ nhờ resume).
4. Chạy tiếp tới hết. Nếu session bị ngắt, chạy lại từ đầu: các mẫu đã có trong `output/vlm_predictions.jsonl` được bỏ qua.
5. Cell 7.4 kiểm tra kết quả (đủ 900 mẫu, 300/category) và tạo file zip để tải về.

> Đây là kết quả **tập con 900 mẫu**, không phải toàn bộ 9.000 mẫu — phải ghi rõ như vậy khi báo cáo.

---
# PHẦN 1: CẤU HÌNH MÔI TRƯỜNG
Kiểm tra GPU, phiên bản Python/PyTorch/CUDA/Transformers; chỉ cài thêm thư viện còn thiếu hoặc quá cũ.

## 1.1 Kiểm tra phiên bản và GPU

In [ ]:
# 1.1 Kiểm tra môi trường: Python, thư viện, GPU
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")  # giảm phân mảnh VRAM
import sys
import platform
from importlib.metadata import version as pkg_version, PackageNotFoundError


def get_pkg_version(pip_name):
    # Lấy phiên bản thư viện bằng metadata (không import, tránh nạp sớm module cũ)
    try:
        return pkg_version(pip_name)
    except PackageNotFoundError:
        return None


print("Python:", sys.version.split()[0], "|", platform.platform())
for pip_name in ["torch", "transformers", "accelerate", "datasets", "huggingface_hub",
                 "pandas", "pyarrow", "Pillow", "matplotlib", "tqdm", "seaborn", "scikit-learn"]:
    print(f"{pip_name:<16}: {get_pkg_version(pip_name) or 'CHƯA CÀI'}")

import torch

print("\nPyTorch:", torch.__version__, "| CUDA build:", torch.version.cuda)
print("torch.cuda.is_available():", torch.cuda.is_available())
if torch.cuda.is_available():
    for gpu_id in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(gpu_id)
        print(f"GPU {gpu_id}: {props.name} | VRAM {props.total_memory / 1024 ** 3:.1f} GB | "
              f"compute capability {props.major}.{props.minor}")
else:
    print("CẢNH BÁO: Không phát hiện GPU. Hãy bật GPU trong Kaggle Settings -> Accelerator. "
          "Chạy trên CPU sẽ cực kỳ chậm với model 4B.")

## 1.2 Cài thư viện
Phi-4-multimodal dùng *remote code* của Microsoft, chạy ổn định với **`transformers==4.48.2`** (model card; nhóm HallusionBench cũng chạy bản này), `accelerate==1.3.0`, `peft==0.13.2`. Bản `transformers` mới hơn (như bản dùng cho Qwen3-VL) có thể lỗi với Phi-4 — nên **chạy notebook này ở session riêng**.

Nếu cell này cài/hạ phiên bản, nó sẽ dừng lại: hãy chọn **Run → Restart Session** rồi chạy lại từ đầu.

In [ ]:
# 1.2 Cài đặt thư viện cho Phi-4-multimodal (BẮT BUỘC transformers==4.48.2)
import subprocess
from packaging.version import Version

PINNED = {"transformers": "4.48.2", "accelerate": "1.3.0", "peft": "0.13.2"}
BASIC = ["datasets", "pandas", "pyarrow", "Pillow", "matplotlib", "tqdm", "seaborn", "scikit-learn"]
PHI4_DEPS = ["soundfile", "scipy", "torchvision", "backoff"]  # remote code của Phi-4 import các gói này

pinned_to_install = [f"{name}=={ver}" for name, ver in PINNED.items() if get_pkg_version(name) != ver]
hub_version = get_pkg_version("huggingface_hub")
if hub_version is None or Version(hub_version) >= Version("1.0"):
    pinned_to_install.append("huggingface_hub<1.0")  # transformers 4.48.2 yêu cầu huggingface_hub < 1.0
other_missing = [name for name in BASIC + PHI4_DEPS if get_pkg_version(name) is None]

if pinned_to_install or other_missing:
    packages = pinned_to_install + other_missing
    print("Cài/đổi phiên bản:", packages)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + packages)
else:
    print("Thư viện đã đúng phiên bản yêu cầu.")

if pinned_to_install:
    raise RuntimeError("Đã đổi phiên bản thư viện. Hãy chọn Run -> Restart Session rồi chạy lại notebook từ đầu "
                       "(cell này sẽ không cài lại vì phiên bản đã đủ).")


## 1.3 Import và cấu hình tham số
Mọi tham số có thể chỉnh nằm trong cell này.

In [ ]:
# 1.3 Import thư viện và khai báo cấu hình
import os
import io
import re
import gc
import sys
import json
import time
import random
import socket
import textwrap
import datetime as dt
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from PIL import Image

# ---------------- Cấu hình dataset ----------------
DATASET_NAME = "lmms-lab-encoder/POPE"
DATA_SOURCE = "hub_parquet"      # "hub_parquet": tải file Parquet từ Hub | "hf_datasets": load_dataset | "local_parquet": đọc file Parquet có sẵn
LOCAL_PARQUET_DIR = None         # Ví dụ: "/kaggle/input/pope-parquet" (dùng khi DATA_SOURCE = "local_parquet")
SPLIT = "auto"                   # "auto" | tên split cụ thể (vd "test") | "all" (gộp mọi file Parquet)

# ---------------- Cấu hình model ----------------
MODEL_NAME = "microsoft/Phi-4-multimodal-instruct"
MODEL_PATH = None                # Ví dụ: "/kaggle/input/phi-4-multimodal-instruct/..." nếu model đã thêm vào Kaggle Input (chạy offline)
DTYPE_OVERRIDE = None            # None = tự chọn (T4/P100 -> float16) | "bfloat16" | "float16" | "float32"
MODEL_REVISION = None            # None = lấy commit mới nhất lúc chạy và GHI LẠI vào summary; có thể gán cứng một commit SHA
ATTN_IMPLEMENTATION = "eager"    # T4/P100 không có FlashAttention2 -> dùng eager (theo model card)
# Phân bổ bộ nhớ: Phi-4 float16 ~ 11 GB trọng số. Với 2 GPU T4: chia GPU0 ~9 GiB, phần còn lại sang GPU1 (không cần offload CPU)
GPU0_MAX_MEMORY = "9GiB"
GPU1_MAX_MEMORY = "13GiB"
SINGLE_GPU_MAX_MEMORY = "13GiB"  # Khi chỉ có 1 GPU
CPU_MAX_MEMORY = "20GiB"         # Chỉ dùng nếu GPU không đủ chỗ (offload CPU làm chậm đáng kể)

# ---------------- Cấu hình inference ----------------
# ---------------- Cấu hình tập mẫu ----------------
USE_SUBSET = True                # True: chạy đúng tập con 900 mẫu cố định (300/category, seed 42) | False: chạy toàn bộ split
SUBSET_NAME = "pope_subset_900_seed42"
MAX_SAMPLES = None               # None = chạy đủ tập đã chọn; số nguyên = chỉ chạy N mẫu ĐẦU (chỉ để thử tốc độ, KHÔNG phải kết quả cuối)
SAMPLING = "stratified_subset" if USE_SUBSET else "full_split"
SEED = 42
BATCH_SIZE = 1                   # Bắt buộc 1 với pipeline Phi-4 này
MAX_NEW_TOKENS = 8               # Câu trả lời chỉ cần 1 từ Yes/No
DO_SAMPLE = False                # False = greedy decoding (tái lập được)
TEMPERATURE = 0.7                # Chỉ có tác dụng khi DO_SAMPLE = True
MAX_IMAGE_SIDE = 1024            # Thu nhỏ ảnh nếu cạnh dài hơn giá trị này (None = giữ nguyên)
OOM_FALLBACK_SIDES = (768, 512, 384)  # Khi gặp CUDA OOM sẽ thử lại với ảnh nhỏ dần

# ---------------- Checkpoint / resume ----------------
SAVE_INTERVAL = 100              # Số mẫu giữa mỗi lần ghi checkpoint
RESUME = True                    # Tiếp tục từ kết quả đã có
RETRY_FAILED = True              # Khi resume: chạy lại các mẫu có status = error
RESUME_FROM = None               # Đường dẫn vlm_predictions.jsonl cũ (vd trong /kaggle/input/...) nếu phiên mới chưa có output

# ---------------- Kiểm tra dữ liệu ----------------
CHECK_ALL_IMAGES = True          # True: giải mã thử toàn bộ ảnh (chậm hơn một chút nhưng đầy đủ)
NUM_PREVIEW = 8                  # Số mẫu hiển thị ảnh

# ---------------- Đường dẫn ----------------
IS_KAGGLE = Path("/kaggle/working").exists()
WORK_DIR = Path("/kaggle/working") if IS_KAGGLE else Path.cwd()
OUTPUT_DIR = WORK_DIR / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CSV_PATH = OUTPUT_DIR / "vlm_predictions.csv"
JSONL_PATH = OUTPUT_DIR / "vlm_predictions.jsonl"
SUMMARY_PATH = OUTPUT_DIR / "inference_summary.json"

assert BATCH_SIZE == 1, "Pipeline Phi-4 chỉ hỗ trợ BATCH_SIZE = 1"

# Cố định seed để tái lập
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("Chạy trên Kaggle:", IS_KAGGLE)
print("Thư mục output  :", OUTPUT_DIR.resolve())

---
# PHẦN 2: TẢI VÀ ĐỌC DATASET
Dataset POPE được lưu dạng Parquet và có thể gồm nhiều file trong cùng một split. Notebook sẽ liệt kê file, chọn split, gộp tất cả file thuộc split đó thành **một** `Dataset` thống nhất (memory-mapped nên không nạp toàn bộ bytes ảnh vào RAM).

## 2.1 Hàm tiện ích: tìm file Parquet theo split

In [ ]:
# 2.1 Các hàm liệt kê / nhóm file Parquet theo split
KNOWN_SPLITS = {"train", "test", "validation", "val", "dev"}


def infer_split_from_path(file_path):
    # Suy ra tên split từ đường dẫn file Parquet (vd data/test-00000-of-00004.parquet -> test)
    path = Path(file_path)
    match = re.match(r"^([A-Za-z_]+)-\d+-of-\d+", path.name)
    if match:
        return match.group(1).lower()
    for part in reversed(path.parts[:-1]):
        if part.lower() in KNOWN_SPLITS:
            return part.lower()
    stem_without_digits = re.sub(r"[-_]?\d+$", "", path.stem)
    return (stem_without_digits or path.stem).lower()


def group_files_by_split(file_paths):
    # Nhóm danh sách file theo split và sắp xếp tên file để thứ tự ổn định
    groups = {}
    for file_path in sorted(str(p) for p in file_paths):
        groups.setdefault(infer_split_from_path(file_path), []).append(file_path)
    return groups


def list_hub_parquet_files(repo_id):
    # Liệt kê toàn bộ file .parquet của dataset trên Hugging Face Hub
    from huggingface_hub import HfApi
    files = HfApi().list_repo_files(repo_id, repo_type="dataset")
    return group_files_by_split([f for f in files if f.lower().endswith(".parquet")])


def list_local_parquet_files(local_dir):
    # Liệt kê file .parquet trong thư mục local (vd Kaggle Input); nhóm theo đường dẫn tương đối
    base = Path(local_dir)
    if not base.exists():
        raise FileNotFoundError(f"Không tìm thấy thư mục: {base}")
    relative_to_abs = {str(p.relative_to(base)): str(p) for p in base.rglob("*.parquet")}
    groups = group_files_by_split(relative_to_abs.keys())
    return {split: [relative_to_abs[rel] for rel in rels] for split, rels in groups.items()}


def resolve_split(available_groups, requested):
    # Chọn split cần chạy; báo lỗi rõ ràng nếu không xác định được
    if not available_groups:
        raise ValueError("Không tìm thấy file Parquet nào.")
    if requested == "all":
        return "all"
    if requested not in (None, "auto"):
        if requested not in available_groups:
            raise ValueError(f"Split '{requested}' không tồn tại. Các split có sẵn: {sorted(available_groups)}")
        return requested
    if len(available_groups) == 1:
        return next(iter(available_groups))
    if "test" in available_groups:
        return "test"
    raise ValueError(f"Có nhiều split {sorted(available_groups)}; hãy đặt SPLIT cụ thể ở cell 1.3.")

## 2.2 Tải dataset và gộp các file Parquet

In [ ]:
# 2.2 Tải dataset POPE, gộp mọi file Parquet của split được chọn thành một Dataset
from datasets import load_dataset, Image as HFImage
from huggingface_hub import hf_hub_download

SPLIT_NAME = None
PARQUET_FILES_USED = []

try:
    if DATA_SOURCE == "hf_datasets":
        # Dùng Hugging Face Dataset API trực tiếp
        from datasets import get_dataset_split_names
        split_names = get_dataset_split_names(DATASET_NAME)
        SPLIT_NAME = resolve_split({name: [] for name in split_names}, SPLIT)
        if SPLIT_NAME == "all":
            raise ValueError("SPLIT='all' chỉ hỗ trợ với hub_parquet / local_parquet.")
        pope_ds = load_dataset(DATASET_NAME, split=SPLIT_NAME)
    else:
        if DATA_SOURCE == "hub_parquet":
            groups = list_hub_parquet_files(DATASET_NAME)
        elif DATA_SOURCE == "local_parquet":
            if not LOCAL_PARQUET_DIR:
                raise ValueError("Hãy đặt LOCAL_PARQUET_DIR khi DATA_SOURCE = 'local_parquet'.")
            groups = list_local_parquet_files(LOCAL_PARQUET_DIR)
        else:
            raise ValueError(f"DATA_SOURCE không hợp lệ: {DATA_SOURCE}")

        print("Các split / số file Parquet phát hiện được:", {k: len(v) for k, v in groups.items()})
        SPLIT_NAME = resolve_split(groups, SPLIT)
        selected_files = sorted(f for files in groups.values() for f in files) if SPLIT_NAME == "all" else groups[SPLIT_NAME]

        if DATA_SOURCE == "hub_parquet":
            local_files = [hf_hub_download(DATASET_NAME, filename=f, repo_type="dataset") for f in selected_files]
        else:
            local_files = selected_files
        PARQUET_FILES_USED = selected_files
        # Gộp tất cả file Parquet thành một Dataset duy nhất (cùng schema)
        pope_ds = load_dataset("parquet", data_files=[str(p) for p in local_files], split="train")
except Exception as exc:
    raise RuntimeError(f"Không tải được dataset: {exc}\n"
                       "Kiểm tra Internet (Settings -> Internet: On) hoặc dùng DATA_SOURCE = 'local_parquet'.") from exc

# Nếu image là Image feature thì tắt tự giải mã: luôn nhận dạng dict {bytes, path}, hàm load_image sẽ xử lý
if "image" in pope_ds.column_names and isinstance(pope_ds.features["image"], HFImage):
    pope_ds = pope_ds.cast_column("image", HFImage(decode=False))

print("\nSplit đang dùng :", SPLIT_NAME)
print("File Parquet    :", PARQUET_FILES_USED if PARQUET_FILES_USED else "(từ load_dataset)")
print("Tổng số mẫu     :", len(pope_ds))

## 2.3 Kiểm tra cấu trúc dataset
Tên cột, kiểu dữ liệu, giá trị null, trùng lặp, nhãn và phân bố theo category.

In [ ]:
# 2.3 Kiểm tra cấu trúc dataset
print("Các cột:", pope_ds.column_names)
print("\nFeatures (kiểu dữ liệu):")
for column_name, feature in pope_ds.features.items():
    print(f"  {column_name:<14}: {feature}")

REQUIRED_COLUMNS = ["question", "image"]
OPTIONAL_COLUMNS = ["id", "question_id", "answer", "category", "image_source"]
missing_required = [c for c in REQUIRED_COLUMNS if c not in pope_ds.column_names]
if missing_required:
    raise ValueError(f"Dataset thiếu cột bắt buộc: {missing_required}")
missing_optional = [c for c in OPTIONAL_COLUMNS if c not in pope_ds.column_names]
if missing_optional:
    print("CẢNH BÁO - thiếu các cột tuỳ chọn:", missing_optional)

# Bảng metadata (không chứa ảnh) để thống kê nhanh
meta_df = pope_ds.remove_columns(["image"]).to_pandas()
print("\nKiểu dữ liệu của metadata:")
print(meta_df.dtypes)

print("\nSố giá trị null theo cột (không gồm image):")
print(meta_df.isna().sum())

if "id" in meta_df.columns:
    print("\nSố ID trùng lặp:", int(meta_df["id"].duplicated().sum()))
if "question_id" in meta_df.columns:
    print("Số question_id trùng lặp:", int(meta_df["question_id"].duplicated().sum()))
print("Số dòng trùng lặp hoàn toàn (không tính ảnh):", int(meta_df.duplicated().sum()))

if "category" in meta_df.columns:
    print("\nSố mẫu theo category:")
    print(meta_df["category"].value_counts(dropna=False))
if "answer" in meta_df.columns:
    # Chỉ để kiểm tra định dạng nhãn; KHÔNG dùng trong inference
    print("\nGiá trị cột answer (chỉ để kiểm tra dữ liệu):")
    print(meta_df["answer"].value_counts(dropna=False))
if "image_source" in meta_df.columns:
    print("\nSố mẫu theo image_source (top 10):")
    print(meta_df["image_source"].value_counts(dropna=False).head(10))

## 2.4 Hàm xử lý ảnh `load_image`
Không giả định mọi ảnh cùng kiểu: hàm xử lý `PIL.Image`, `dict {bytes, path}`, `bytes` thuần và đường dẫn file; luôn trả về PIL Image RGB hoặc ném `ImageLoadError` có thông điệp rõ ràng.

In [ ]:
# 2.4 Hàm xử lý ảnh: chuẩn hoá mọi dạng dữ liệu ảnh về PIL Image RGB
class ImageLoadError(Exception):
    # Lỗi khi không thể đọc / giải mã ảnh
    pass


def load_image(image_data):
    # Nhận dữ liệu ảnh từ dataset (dict bytes/path, PIL Image, bytes, đường dẫn) và trả về PIL Image RGB
    try:
        if image_data is None:
            raise ValueError("dữ liệu ảnh là None")
        if isinstance(image_data, Image.Image):
            image = image_data
        elif isinstance(image_data, dict):
            raw_bytes = image_data.get("bytes")
            image_path = image_data.get("path")
            if raw_bytes:
                image = Image.open(io.BytesIO(raw_bytes))
            elif image_path and Path(image_path).is_file():
                image = Image.open(image_path)
            else:
                raise ValueError(f"dict không có bytes và path không hợp lệ (path={image_path!r})")
        elif isinstance(image_data, (bytes, bytearray)):
            image = Image.open(io.BytesIO(bytes(image_data)))
        elif isinstance(image_data, (str, Path)):
            if not Path(image_data).is_file():
                raise ValueError(f"không tìm thấy file ảnh: {image_data}")
            image = Image.open(image_data)
        else:
            raise TypeError(f"kiểu dữ liệu ảnh không được hỗ trợ: {type(image_data).__name__}")
        image.load()  # Ép giải mã thật sự để phát hiện ảnh hỏng
        return image.convert("RGB")
    except Exception as exc:
        raise ImageLoadError(f"{type(exc).__name__}: {exc}") from exc

## 2.5 Kiểm tra ảnh có giải mã được không

In [ ]:
# 2.5 Kiểm tra null / ảnh lỗi: giải mã thử ảnh bằng load_image
def scan_images(dataset, check_all=True, preview_limit=64, batch_size=64):
    # Trả về (số ảnh đã kiểm tra, danh sách (row_index, lỗi))
    limit = len(dataset) if check_all else min(preview_limit, len(dataset))
    bad_rows = []
    checked = 0
    image_only = dataset.select_columns(["image"])
    for batch in tqdm(image_only.iter(batch_size=batch_size), total=(len(dataset) + batch_size - 1) // batch_size,
                      desc="Kiểm tra ảnh"):
        for image_data in batch["image"]:
            if checked >= limit:
                break
            try:
                load_image(image_data)
            except ImageLoadError as exc:
                bad_rows.append((checked, str(exc)))
            checked += 1
        if checked >= limit:
            break
    return checked, bad_rows


checked_count, bad_image_rows = scan_images(pope_ds, check_all=CHECK_ALL_IMAGES)
print(f"Đã kiểm tra {checked_count} ảnh; không giải mã được: {len(bad_image_rows)}")
if bad_image_rows:
    print("Một số dòng lỗi (row_index, lỗi):", bad_image_rows[:10])
    print("Các mẫu này vẫn được giữ lại và sẽ có status = error khi inference (không bị bỏ âm thầm).")

## 2.6 Hiển thị mẫu dữ liệu (ảnh thực tế)
Chọn mẫu rải đều các category. Ground truth chỉ hiển thị ở đây để người nghiên cứu quan sát dữ liệu, **không** đưa vào model.

In [ ]:
# 2.6 Hiển thị bảng metadata và ảnh thực tế của một số mẫu
display_columns = [c for c in ["id", "question_id", "question", "answer", "category", "image_source"] if c in meta_df.columns]
try:
    display(meta_df[display_columns].head(10))
except NameError:
    print(meta_df[display_columns].head(10).to_string())


def pick_preview_indices(meta, n, seed):
    # Chọn n chỉ số, ưu tiên rải đều theo category
    rng = np.random.default_rng(seed)
    if "category" in meta.columns and meta["category"].notna().any():
        groups = [np.array(list(idx)) for _, idx in meta.groupby("category").groups.items()]
        per_group = max(1, n // len(groups))
        chosen = []
        for group_indices in groups:
            chosen.extend(rng.choice(group_indices, size=min(per_group, len(group_indices)), replace=False).tolist())
        return chosen[:n]
    return rng.choice(len(meta), size=min(n, len(meta)), replace=False).tolist()


def show_samples(dataset, meta, indices, cols=4):
    # Vẽ lưới ảnh kèm câu hỏi và nhãn ground truth (chỉ để kiểm tra dữ liệu)
    rows = (len(indices) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(4.2 * cols, 4.2 * rows))
    axes = np.array(axes).reshape(-1)
    for axis in axes:
        axis.axis("off")
    for axis, row_index in zip(axes, indices):
        try:
            axis.imshow(load_image(dataset[int(row_index)]["image"]))
        except ImageLoadError as exc:
            axis.text(0.5, 0.5, f"Ảnh lỗi\n{exc}", ha="center", va="center", wrap=True)
        row = meta.iloc[int(row_index)]
        title = f"#{row_index} | {row.get('category', '')} | GT: {row.get('answer', '')}\n" + \
                textwrap.fill(str(row["question"]), 40)
        axis.set_title(title, fontsize=8)
    plt.tight_layout()
    plt.show()


preview_indices = pick_preview_indices(meta_df, NUM_PREVIEW, SEED)
show_samples(pope_ds, meta_df, preview_indices)

## 2.7 Khoá định danh mẫu và chọn tập mẫu cần chạy
Khoá `sample_key` (= `id`) dùng cho checkpoint/resume. Tập chạy là **900 mẫu cố định** (300 mỗi category, seed 42) lấy từ file `benchmarks/pope/pope_subset_900_seed42.json` — danh sách được nhúng sẵn bên dưới để notebook không cần file ngoài. Cả 3 model phải chạy đúng danh sách này thì mới so sánh được.

> Lưu ý: POPE sắp xếp theo category (dòng 0–2999 `adversarial`, 3000–5999 `popular`, 6000–8999 `random`), nên **không** được lấy N mẫu đầu để thay cho tập con này.

In [ ]:
# 2.7 Tạo khoá định danh mẫu và chọn tập mẫu chạy (tập con cố định 900 mẫu)
import hashlib


def build_sample_keys(meta):
    # Trả về (danh sách khoá dạng chuỗi, tên cách tạo khoá)
    has_id = "id" in meta.columns and meta["id"].notna().all()
    if has_id and meta["id"].astype(str).is_unique:
        return meta["id"].astype(str).tolist(), "id"
    if has_id and "question_id" in meta.columns and meta["question_id"].notna().all():
        composite = meta["id"].astype(str) + "|" + meta["question_id"].astype(str)
        if composite.is_unique:
            return composite.tolist(), "id|question_id"
    return [f"row{i}" for i in range(len(meta))], "row_index"


def build_stratified_indices(meta, per_category, seed):
    # Cách đã dùng để sinh danh sách bên dưới: mỗi category lấy ngẫu nhiên per_category dòng (numpy default_rng)
    rng = np.random.default_rng(seed)
    chosen = []
    for _, idx in meta.groupby("category").groups.items():
        idx = np.array(sorted(idx))
        chosen.extend(rng.choice(idx, size=per_category, replace=False).tolist())
    return sorted(chosen)


# Danh sách id của tập con (nguồn: benchmarks/pope/pope_subset_900_seed42.json)
SUBSET_IDS_BY_CATEGORY = {
    "adversarial": [
        "20", "57", "64", "65", "67", "87", "98", "104", "120", "122", "166", "174", "186", "193",
        "211", "223", "231", "232", "241", "246", "248", "251", "253", "255", "256", "258", "270",
        "278", "281", "304", "305", "322", "341", "348", "360", "372", "391", "393", "412", "425",
        "427", "438", "447", "450", "452", "464", "478", "479", "480", "497", "505", "517", "518",
        "525", "529", "535", "545", "555", "560", "591", "609", "620", "630", "666", "670", "689",
        "692", "758", "760", "766", "772", "778", "786", "793", "798", "802", "806", "822", "837",
        "853", "862", "870", "871", "875", "888", "895", "901", "907", "916", "920", "959", "972",
        "974", "982", "986", "992", "1001", "1009", "1013", "1025", "1029", "1047", "1062", "1082",
        "1097", "1100", "1136", "1161", "1163", "1171", "1186", "1197", "1210", "1214", "1216",
        "1224", "1225", "1230", "1239", "1240", "1244", "1269", "1283", "1285", "1287", "1291",
        "1300", "1304", "1309", "1315", "1316", "1320", "1324", "1326", "1328", "1330", "1331",
        "1357", "1361", "1371", "1390", "1394", "1415", "1423", "1427", "1429", "1433", "1435",
        "1437", "1443", "1446", "1451", "1458", "1464", "1488", "1506", "1516", "1559", "1563",
        "1567", "1568", "1584", "1591", "1597", "1600", "1601", "1620", "1625", "1632", "1695",
        "1708", "1710", "1718", "1723", "1730", "1739", "1753", "1755", "1756", "1769", "1782",
        "1796", "1818", "1822", "1844", "1847", "1851", "1862", "1863", "1866", "1885", "1886",
        "1888", "1890", "1901", "1904", "1909", "1913", "1921", "1947", "1948", "1950", "1952",
        "1976", "1981", "1996", "2001", "2003", "2008", "2031", "2050", "2057", "2062", "2065",
        "2076", "2078", "2087", "2090", "2091", "2110", "2112", "2129", "2134", "2140", "2141",
        "2148", "2163", "2190", "2195", "2204", "2208", "2216", "2221", "2225", "2232", "2235",
        "2241", "2244", "2246", "2253", "2256", "2266", "2283", "2298", "2318", "2320", "2322",
        "2323", "2330", "2332", "2340", "2349", "2350", "2366", "2415", "2428", "2439", "2454",
        "2505", "2510", "2511", "2524", "2546", "2547", "2592", "2614", "2618", "2635", "2636",
        "2645", "2665", "2674", "2702", "2742", "2778", "2789", "2791", "2800", "2818", "2824",
        "2839", "2845", "2849", "2850", "2854", "2856", "2862", "2865", "2879", "2904", "2909",
        "2922", "2928", "2934", "2947", "2956", "2960", "2969", "2990", "2993",
    ],
    "popular": [
        "3014", "3016", "3022", "3035", "3042", "3047", "3070", "3084", "3093", "3097", "3106",
        "3119", "3126", "3128", "3140", "3152", "3158", "3163", "3166", "3169", "3200", "3210",
        "3234", "3246", "3248", "3251", "3256", "3263", "3264", "3266", "3276", "3299", "3310",
        "3321", "3331", "3338", "3341", "3351", "3353", "3356", "3362", "3363", "3365", "3366",
        "3371", "3380", "3382", "3392", "3399", "3401", "3419", "3421", "3449", "3460", "3463",
        "3470", "3471", "3480", "3488", "3500", "3503", "3557", "3559", "3577", "3582", "3590",
        "3599", "3634", "3644", "3664", "3678", "3704", "3706", "3708", "3721", "3724", "3769",
        "3780", "3795", "3809", "3813", "3827", "3849", "3850", "3855", "3874", "3876", "3892",
        "3895", "3902", "3914", "3922", "3947", "4000", "4002", "4034", "4036", "4037", "4045",
        "4080", "4095", "4133", "4136", "4137", "4180", "4185", "4186", "4219", "4242", "4249",
        "4251", "4255", "4265", "4283", "4301", "4309", "4314", "4320", "4326", "4329", "4337",
        "4339", "4346", "4355", "4375", "4378", "4380", "4381", "4387", "4393", "4451", "4454",
        "4458", "4468", "4470", "4471", "4497", "4501", "4515", "4524", "4526", "4564", "4573",
        "4583", "4584", "4585", "4607", "4613", "4633", "4646", "4651", "4652", "4655", "4665",
        "4691", "4698", "4703", "4726", "4727", "4735", "4737", "4746", "4757", "4764", "4788",
        "4796", "4804", "4823", "4832", "4836", "4839", "4862", "4863", "4872", "4876", "4878",
        "4901", "4904", "4910", "4922", "4931", "4943", "4945", "4980", "4982", "4993", "4994",
        "4998", "5012", "5013", "5015", "5019", "5030", "5039", "5047", "5063", "5071", "5073",
        "5075", "5087", "5091", "5096", "5105", "5107", "5113", "5115", "5124", "5128", "5145",
        "5150", "5167", "5172", "5184", "5194", "5195", "5216", "5219", "5220", "5221", "5232",
        "5235", "5237", "5245", "5251", "5260", "5263", "5268", "5271", "5281", "5289", "5322",
        "5324", "5334", "5343", "5366", "5371", "5374", "5383", "5386", "5389", "5418", "5428",
        "5433", "5434", "5441", "5450", "5487", "5508", "5509", "5511", "5513", "5529", "5530",
        "5538", "5545", "5552", "5561", "5591", "5600", "5602", "5611", "5620", "5627", "5628",
        "5640", "5641", "5644", "5668", "5670", "5672", "5677", "5678", "5682", "5718", "5736",
        "5741", "5749", "5767", "5782", "5793", "5806", "5817", "5834", "5845", "5848", "5850",
        "5854", "5855", "5867", "5874", "5893", "5901", "5902", "5918", "5933", "5935", "5946",
        "5957", "5973", "5977",
    ],
    "random": [
        "6003", "6004", "6012", "6022", "6050", "6072", "6082", "6087", "6102", "6105", "6112",
        "6121", "6134", "6136", "6140", "6151", "6152", "6172", "6176", "6179", "6192", "6193",
        "6201", "6203", "6233", "6235", "6240", "6284", "6292", "6294", "6299", "6303", "6304",
        "6310", "6323", "6331", "6332", "6334", "6339", "6384", "6389", "6411", "6420", "6451",
        "6452", "6485", "6502", "6515", "6530", "6531", "6543", "6578", "6612", "6616", "6624",
        "6627", "6652", "6654", "6659", "6665", "6667", "6678", "6687", "6700", "6715", "6727",
        "6730", "6735", "6736", "6748", "6771", "6776", "6781", "6782", "6787", "6816", "6820",
        "6822", "6824", "6841", "6842", "6865", "6867", "6868", "6870", "6885", "6894", "6899",
        "6926", "6937", "6938", "6940", "6948", "6968", "6979", "6983", "6986", "6997", "7000",
        "7005", "7009", "7023", "7032", "7045", "7064", "7095", "7115", "7119", "7120", "7125",
        "7133", "7136", "7141", "7159", "7166", "7174", "7197", "7204", "7209", "7228", "7229",
        "7238", "7240", "7242", "7250", "7255", "7266", "7274", "7275", "7278", "7280", "7291",
        "7292", "7293", "7300", "7336", "7344", "7347", "7378", "7379", "7391", "7393", "7395",
        "7415", "7420", "7439", "7449", "7461", "7465", "7474", "7486", "7492", "7506", "7509",
        "7513", "7518", "7520", "7528", "7532", "7533", "7552", "7558", "7590", "7619", "7636",
        "7650", "7652", "7664", "7676", "7691", "7697", "7706", "7711", "7713", "7722", "7736",
        "7743", "7761", "7769", "7783", "7791", "7797", "7811", "7822", "7825", "7853", "7874",
        "7885", "7886", "7894", "7896", "7910", "7919", "7921", "7953", "7962", "7967", "7973",
        "7981", "7992", "7997", "8000", "8006", "8033", "8051", "8058", "8059", "8060", "8076",
        "8094", "8117", "8122", "8129", "8133", "8149", "8150", "8165", "8168", "8181", "8187",
        "8194", "8201", "8202", "8215", "8231", "8245", "8275", "8287", "8289", "8294", "8309",
        "8310", "8341", "8354", "8363", "8375", "8385", "8397", "8398", "8403", "8407", "8437",
        "8446", "8449", "8480", "8505", "8513", "8514", "8527", "8530", "8531", "8542", "8544",
        "8553", "8555", "8575", "8577", "8585", "8588", "8612", "8614", "8620", "8629", "8641",
        "8644", "8645", "8653", "8661", "8663", "8669", "8687", "8691", "8708", "8752", "8760",
        "8765", "8766", "8770", "8776", "8795", "8810", "8815", "8826", "8831", "8834", "8841",
        "8844", "8848", "8908", "8913", "8929", "8931", "8940", "8956", "8958", "8965", "8976",
        "8983", "8984", "8999",
    ],
}
SUBSET_IDS_SHA256 = "75676fd7ecaffd366aa6b5d50195ae23f2281b07d834d06e82754ae3a0caddd6"

sample_keys, SAMPLE_KEY_MODE = build_sample_keys(meta_df)
meta_records = meta_df.to_dict("records")  # Truy cập nhanh theo chỉ số dòng
key_to_index = {k: i for i, k in enumerate(sample_keys)}

if USE_SUBSET:
    assert SAMPLE_KEY_MODE == "id", f"Tập con được định nghĩa theo id, nhưng khoá mẫu hiện là {SAMPLE_KEY_MODE}"
    all_subset_ids = [k for ids in SUBSET_IDS_BY_CATEGORY.values() for k in ids]
    assert len(all_subset_ids) == len(set(all_subset_ids)) == 900, "Danh sách tập con phải có đúng 900 id duy nhất"
    sha = hashlib.sha256("\n".join(sorted(all_subset_ids, key=int)).encode()).hexdigest()
    assert sha == SUBSET_IDS_SHA256, "Danh sách id nhúng trong notebook đã bị sửa so với bản gốc"
    missing_ids = [k for k in all_subset_ids if k not in key_to_index]
    assert not missing_ids, f"Dataset thiếu {len(missing_ids)} id của tập con, ví dụ: {missing_ids[:5]}"
    # Kiểm tra mỗi id nằm đúng category như khi tạo tập con
    wrong_category = [(k, cat, meta_records[key_to_index[k]].get("category"))
                      for cat, ids in SUBSET_IDS_BY_CATEGORY.items() for k in ids
                      if meta_records[key_to_index[k]].get("category") != cat]
    assert not wrong_category, f"Lệch category so với tập con: {wrong_category[:5]} -> dataset khác bản đã dùng"
    run_indices = sorted(key_to_index[k] for k in all_subset_ids)
    regenerated = build_stratified_indices(meta_df, 300, SEED)
    print("Tái sinh bằng numpy trên dataset này khớp danh sách nhúng:", regenerated == run_indices,
          "(danh sách nhúng là bản chuẩn, kết quả này chỉ để tham khảo)")
else:
    run_indices = list(range(len(pope_ds)))

if MAX_SAMPLES is not None:
    run_indices = run_indices[:MAX_SAMPLES]
    print(f"CẢNH BÁO: MAX_SAMPLES = {MAX_SAMPLES} -> chỉ chạy thử, KHÔNG phải kết quả cuối.")

print("Cách tạo khoá mẫu:", SAMPLE_KEY_MODE)
print(f"Số mẫu sẽ chạy: {len(run_indices)} / {len(pope_ds)} (SAMPLING = {SAMPLING})")
print("Phân bố category của tập chạy:\n", meta_df.iloc[run_indices]["category"].value_counts())
if "answer" in meta_df.columns:
    # Chỉ để kiểm tra phân bố nhãn; KHÔNG dùng trong inference
    print("Phân bố nhãn (chỉ để kiểm tra dữ liệu):\n", meta_df.iloc[run_indices]["answer"].value_counts())


---
# PHẦN 3: TẢI MODEL PHI-4-MULTIMODAL-INSTRUCT
Dùng `AutoModelForCausalLM` + `AutoProcessor` với `trust_remote_code=True` (Phi-4 dùng code của Microsoft), `attn_implementation="eager"`. Dtype: `bfloat16` nếu GPU từ Ampere trở lên, ngược lại `float16` (T4/P100 không có bf16 gốc). Không dùng quantization. Commit của model được **ghim và ghi lại** vào summary.

In [ ]:
# 3.1 Kiểm tra phiên bản transformers trước khi tải model
import transformers
from transformers import AutoProcessor, AutoModelForCausalLM, AutoConfig

REQUIRED_TRANSFORMERS = "4.48.2"
print("transformers:", transformers.__version__)
if transformers.__version__ != REQUIRED_TRANSFORMERS:
    raise RuntimeError(f"Phi-4 cần transformers=={REQUIRED_TRANSFORMERS} (đang là {transformers.__version__}). "
                       "Chạy lại cell 1.2 rồi Restart Session; nếu vẫn sai, mở notebook ở session mới.")
print("transformers OK cho Phi-4-multimodal")


In [ ]:
# 3.2 Tải model và processor (chỉ tải một lần)
def select_dtype():
    # Chọn dtype phù hợp với GPU
    if DTYPE_OVERRIDE:
        return getattr(torch, DTYPE_OVERRIDE)
    if not torch.cuda.is_available():
        return torch.float32
    major, _ = torch.cuda.get_device_capability(0)
    return torch.bfloat16 if major >= 8 else torch.float16


def build_max_memory():
    # Giới hạn bộ nhớ từng thiết bị cho accelerate (device_map="auto")
    gpu_count = torch.cuda.device_count() if torch.cuda.is_available() else 0
    if gpu_count >= 2:
        return {0: GPU0_MAX_MEMORY, 1: GPU1_MAX_MEMORY, "cpu": CPU_MAX_MEMORY}
    if gpu_count == 1:
        return {0: SINGLE_GPU_MAX_MEMORY, "cpu": CPU_MAX_MEMORY}
    return None


def resolve_commit_sha(repo_id, repo_type="model"):
    # Lấy commit SHA hiện tại của repo trên Hub (None nếu không truy cập được)
    try:
        from huggingface_hub import HfApi
        api = HfApi()
        info = api.model_info(repo_id) if repo_type == "model" else api.dataset_info(repo_id)
        return getattr(info, "sha", None)
    except Exception as exc:
        print(f"Không lấy được commit SHA của {repo_id}: {exc}")
        return None


MODEL_REVISION_RESOLVED = None
DEVICE_ALLOCATION = None


def load_model_and_processor():
    # Tải Phi-4-multimodal và processor với xử lý lỗi rõ ràng
    global MODEL_REVISION_RESOLVED, DEVICE_ALLOCATION
    source = MODEL_PATH or MODEL_NAME
    revision = None if MODEL_PATH else (MODEL_REVISION or resolve_commit_sha(MODEL_NAME))
    MODEL_REVISION_RESOLVED = revision
    rev_kwargs = {"revision": revision} if revision else {}
    dtype = select_dtype()
    max_memory = build_max_memory()
    device_map = "auto" if torch.cuda.is_available() else None
    print(f"Nguồn model: {source} | revision: {revision} | dtype: {dtype} | attn: {ATTN_IMPLEMENTATION} | "
          f"device_map: {device_map} | max_memory: {max_memory}")
    try:
        config = AutoConfig.from_pretrained(source, trust_remote_code=True, **rev_kwargs)
        config._attn_implementation = ATTN_IMPLEMENTATION
        config._attn_implementation_autoset = False
        loaded_processor = AutoProcessor.from_pretrained(source, trust_remote_code=True, **rev_kwargs)
        model_kwargs = dict(config=config, torch_dtype=dtype, device_map=device_map, trust_remote_code=True,
                            attn_implementation=ATTN_IMPLEMENTATION, low_cpu_mem_usage=True, **rev_kwargs)
        if max_memory:
            model_kwargs["max_memory"] = max_memory
        loaded_model = AutoModelForCausalLM.from_pretrained(source, **model_kwargs)
    except torch.cuda.OutOfMemoryError as exc:
        raise RuntimeError("CUDA OOM khi tải model. Hãy Restart Session, chọn GPU T4 x2 "
                           "hoặc đảm bảo không có tiến trình khác dùng GPU.") from exc
    except OSError as exc:
        raise RuntimeError(f"Không tải được model '{source}': {exc}\n"
                           "Kiểm tra Internet (Settings -> Internet: On) hoặc đặt MODEL_PATH tới model trong Kaggle Input.") from exc
    loaded_model.eval()
    hf_map = getattr(loaded_model, "hf_device_map", None)
    DEVICE_ALLOCATION = {str(k): str(v) for k, v in hf_map.items()} if hf_map else str(next(loaded_model.parameters()).device)
    return loaded_model, loaded_processor


if globals().get("model") is not None and globals().get("processor") is not None:
    print("Model và processor đã được tải trước đó -> không tải lại.")
else:
    model, processor = load_model_and_processor()

# Thông tin model sau khi tải
num_params = sum(p.numel() for p in model.parameters())
print(f"\nLớp model : {type(model).__name__}")
print(f"Số tham số: {num_params / 1e9:.2f} B")
print(f"dtype     : {model.dtype} | device: {model.device}")
if torch.cuda.is_available():
    for gpu_id in range(torch.cuda.device_count()):
        print(f"GPU {gpu_id} VRAM đã cấp phát: {torch.cuda.memory_allocated(gpu_id) / 1024 ** 3:.2f} GB")
if isinstance(DEVICE_ALLOCATION, dict):
    placement = {}
    for device in DEVICE_ALLOCATION.values():
        placement[device] = placement.get(device, 0) + 1
    print("Số module theo thiết bị:", placement)
    if any(d in ("cpu", "disk") for d in placement):
        print("CẢNH BÁO: một phần model nằm trên CPU/disk (offload) -> inference sẽ chậm hơn nhiều.")


---
# PHẦN 4: THIẾT KẾ PROMPT INFERENCE
**`PROMPT_TEMPLATE` và `normalize_answer` giữ NGUYÊN như notebook Qwen3** để ba model so sánh được với nhau. Chỉ khác cách đưa ảnh vào: Phi-4 dùng định dạng `<|user|><|image_1|>{prompt}<|end|><|assistant|>` thay vì chat template của Qwen.

Chuẩn hoá đầu ra: lấy từ đầu tiên của câu trả lời (bỏ dấu câu, không phân biệt hoa thường). Nếu không phải `yes`/`no` thì gán `Invalid` và `status = invalid_output` — **không** âm thầm chuyển thành Yes/No.

In [ ]:
# 4.1 Prompt và hàm chuẩn hoá đầu ra
PROMPT_TEMPLATE = (
    "Answer the following question based only on the provided image.\n\n"
    "Question: {question}\n\n"
    "Respond with exactly one word: Yes or No.\n\n"
    "Do not provide explanations."
)

STATUS_SUCCESS = "success"
STATUS_INVALID = "invalid_output"
STATUS_ERROR = "error"


def build_prompt(question):
    # Tạo nội dung prompt từ câu hỏi (chỉ dùng question)
    return PROMPT_TEMPLATE.format(question=str(question).strip())


def build_phi4_text(question):
    # Định dạng hội thoại của Phi-4-multimodal: 1 ảnh (<|image_1|>) + 1 đoạn text
    return f"<|user|><|image_1|>{build_prompt(question)}<|end|><|assistant|>"


def normalize_answer(raw_text):
    # Chuẩn hoá chuỗi sinh ra thành (nhãn, status); nhãn là Yes / No / Invalid
    cleaned = re.sub(r"^[\W_]+", "", str(raw_text).strip().lower())
    match = re.match(r"(yes|no)\b", cleaned)
    if match:
        return match.group(1).capitalize(), STATUS_SUCCESS
    return "Invalid", STATUS_INVALID


# Kiểm thử nhanh hàm chuẩn hoá trên các chuỗi mẫu
for sample_text in ["Yes", "no", " No.", "Yes, there is", "Maybe", "", "Nobody", "Not sure"]:
    print(f"{sample_text!r:<16} -> {normalize_answer(sample_text)}")
print("\nPrompt mẫu (đầy đủ định dạng Phi-4):\n" + build_phi4_text("Is there a dog in the image?"))

---
# PHẦN 5: XÂY DỰNG HÀM INFERENCE
- `run_inference(question, image)`: một mẫu, có xử lý lỗi riêng, đo thời gian (`latency_ms`) và tự thử lại với ảnh nhỏ hơn khi gặp CUDA OOM.
- `run_inference_batch(...)`: giữ tên hàm như notebook Qwen3 nhưng chỉ chạy tuần tự từng mẫu (`BATCH_SIZE = 1`).

Hàm chỉ nhận `question` và `image` — không có tham số nào cho ground truth.

In [ ]:
# 5.1 Hàm inference cho một mẫu
def free_memory():
    # Giải phóng bộ nhớ CPU/GPU không cần thiết
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def resize_if_needed(image, max_side):
    # Thu nhỏ ảnh nếu cạnh dài vượt max_side để tiết kiệm VRAM
    width, height = image.size
    longest = max(width, height)
    if max_side and longest > max_side:
        scale = max_side / longest
        return image.resize((max(1, int(width * scale)), max(1, int(height * scale))), Image.BICUBIC)
    return image


def make_result(prediction, raw_output, status, error, latency_ms=None):
    # Đóng gói kết quả của một mẫu
    return {"prediction": prediction, "raw_output": raw_output, "status": status, "error": error,
            "latency_ms": latency_ms}


def generate_answers(questions, images):
    # Sinh câu trả lời cho (question, PIL image); Phi-4 chạy từng mẫu một (batch = 1); trả về danh sách chuỗi
    if len(questions) != 1 or len(images) != 1:
        raise ValueError("Pipeline Phi-4 chỉ hỗ trợ batch_size = 1")
    text = build_phi4_text(questions[0])
    inputs = processor(text=text, images=[images[0]], return_tensors="pt")
    first_device = model.device
    moved = {key: (value.to(first_device) if torch.is_tensor(value) else value) for key, value in inputs.items()}

    generation_kwargs = {"max_new_tokens": MAX_NEW_TOKENS, "do_sample": DO_SAMPLE}
    if DO_SAMPLE:
        generation_kwargs["temperature"] = TEMPERATURE

    output_ids = None
    try:
        with torch.inference_mode():
            output_ids = model.generate(**moved, **generation_kwargs)
        prompt_length = int(moved["input_ids"].shape[-1])
        new_tokens = output_ids[:, prompt_length:]  # Bỏ phần prompt, chỉ giữ token mới sinh
        return processor.batch_decode(new_tokens, skip_special_tokens=True, clean_up_tokenization_spaces=False)
    finally:
        del moved, inputs
        if output_ids is not None:
            del output_ids


def run_inference(question, image):
    # Chạy VLM cho một mẫu; trả về dict(prediction, raw_output, status, error, latency_ms)
    try:
        pil_image = load_image(image)
    except ImageLoadError as exc:
        return make_result(None, None, STATUS_ERROR, f"image_error: {exc}")

    side_candidates = [MAX_IMAGE_SIDE] + [s for s in OOM_FALLBACK_SIDES if MAX_IMAGE_SIDE is None or s < MAX_IMAGE_SIDE]
    last_error = "unknown"
    for max_side in side_candidates:
        started = time.perf_counter()
        try:
            raw_text = generate_answers([question], [resize_if_needed(pil_image, max_side)])[0]
            latency_ms = round((time.perf_counter() - started) * 1000, 1)
            label, status = normalize_answer(raw_text)
            return make_result(label, raw_text.strip(), status, None, latency_ms)
        except torch.cuda.OutOfMemoryError:
            free_memory()
            last_error = f"CUDA OOM (max_side={max_side})"  # Thử lại với ảnh nhỏ hơn
        except Exception as exc:
            free_memory()
            return make_result(None, None, STATUS_ERROR, f"{type(exc).__name__}: {exc}")
    return make_result(None, None, STATUS_ERROR, last_error)


def run_inference_batch(questions, images):
    # Giữ tên hàm như notebook Qwen3; Phi-4 xử lý tuần tự từng mẫu
    return [run_inference(q, img) for q, img in zip(questions, images)]


In [ ]:
# 5.2 Chạy thử vài mẫu (rải đều tập chạy) để kiểm tra pipeline và ước lượng thời gian
SMOKE_N = 5
step = max(1, len(run_indices) // SMOKE_N)
smoke_indices = run_indices[::step][:SMOKE_N]
smoke_latencies = []
for row_index in smoke_indices:
    smoke_result = run_inference(meta_records[row_index]["question"], pope_ds[int(row_index)]["image"])
    print(f"#{row_index} | {meta_records[row_index].get('category')} | Q: {meta_records[row_index]['question']}")
    print("   ->", smoke_result)
    if smoke_result["latency_ms"] is not None:
        smoke_latencies.append(smoke_result["latency_ms"])
free_memory()

if len(smoke_latencies) >= 2:
    warm = smoke_latencies[1:]  # bỏ mẫu đầu (khởi động)
    per_sample = sum(warm) / len(warm) / 1000
    print(f"\nTrung bình ~{per_sample:.2f} giây/mẫu -> ước lượng {len(run_indices)} mẫu: ~{per_sample * len(run_indices) / 60:.0f} phút "
          "(chỉ là ước lượng từ vài mẫu).")


---
# PHẦN 6: CHẠY INFERENCE TRÊN DATASET
Mỗi mẫu được ghi thành một dòng JSON (append) vào `vlm_predictions.jsonl` theo từng lô `SAVE_INTERVAL` mẫu (checkpoint). Khi chạy lại với `RESUME = True`:
- các mẫu đã hoàn thành (`success` / `invalid_output`) được bỏ qua;
- các mẫu `error` được chạy lại nếu `RETRY_FAILED = True`; bản ghi lỗi cũ bị loại khỏi file trước khi ghi tiếp nên **không bị trùng**.

Nếu phiên Kaggle mới chưa có output cũ, hãy thêm output phiên trước làm Input và đặt `RESUME_FROM` tới file `.jsonl` của nó.

In [ ]:
# 6.1 Hàm đọc / ghi checkpoint và tạo bản ghi kết quả
OUTPUT_COLUMNS = ["id", "question_id", "question", "prediction", "image_source", "category", "status", "error"]
EXTRA_COLUMNS = ["raw_output", "row_index", "sample_key", "latency_ms"]  # Cột phụ để truy vết (evaluation không bắt buộc)


def json_default(value):
    # Chuyển kiểu numpy sang kiểu Python để ghi JSON
    if isinstance(value, np.generic):
        return value.item()
    return str(value)


def clean_value(value):
    # NaN/None -> None; numpy scalar -> Python scalar
    if value is None:
        return None
    if isinstance(value, np.generic):
        value = value.item()
    if isinstance(value, float) and np.isnan(value):
        return None
    return value


def build_record(row_index, result):
    # Tạo bản ghi cho một mẫu (không chứa bytes ảnh)
    meta = meta_records[row_index]
    return {
        "id": clean_value(meta.get("id")),
        "question_id": clean_value(meta.get("question_id")),
        "question": clean_value(meta.get("question")),
        "prediction": result["prediction"],
        "image_source": clean_value(meta.get("image_source")),
        "category": clean_value(meta.get("category")),
        "status": result["status"],
        "error": result["error"],
        "raw_output": result["raw_output"],
        "row_index": int(row_index),
        "sample_key": sample_keys[row_index],
        "latency_ms": result.get("latency_ms"),
    }


def read_jsonl_records(path):
    # Đọc JSONL thành dict sample_key -> bản ghi (bản ghi sau ghi đè bản ghi trước); bỏ qua dòng hỏng
    records, broken_lines = {}, 0
    if path is None or not Path(path).exists():
        return records, broken_lines
    with open(path, "r", encoding="utf-8") as file:
        for line in file:
            line = line.strip()
            if not line:
                continue
            try:
                record = json.loads(line)
                records[record["sample_key"]] = record
            except (json.JSONDecodeError, KeyError):
                broken_lines += 1  # Dòng cuối có thể bị cắt khi notebook bị ngắt đột ngột
    return records, broken_lines


def write_jsonl_atomic(path, records):
    # Ghi lại toàn bộ file JSONL qua file tạm rồi thay thế (an toàn khi bị ngắt giữa chừng)
    temp_path = Path(str(path) + ".tmp")
    with open(temp_path, "w", encoding="utf-8") as file:
        for record in records:
            file.write(json.dumps(record, ensure_ascii=False, default=json_default) + "\n")
    temp_path.replace(path)


def append_jsonl(path, records):
    # Ghi nối thêm các bản ghi mới (checkpoint)
    with open(path, "a", encoding="utf-8") as file:
        for record in records:
            file.write(json.dumps(record, ensure_ascii=False, default=json_default) + "\n")
        file.flush()
        os.fsync(file.fileno())

In [ ]:
# 6.2 Chuẩn bị resume: xác định các mẫu đã xong và các mẫu còn phải chạy
run_keys = {sample_keys[i] for i in run_indices}

resume_source = None
if RESUME:
    if JSONL_PATH.exists():
        resume_source = JSONL_PATH
    elif RESUME_FROM and Path(RESUME_FROM).exists():
        resume_source = Path(RESUME_FROM)

existing_records, broken_line_count = read_jsonl_records(resume_source) if resume_source else ({}, 0)
done_statuses = {STATUS_SUCCESS, STATUS_INVALID}
kept_records = [r for r in existing_records.values()
                if (r.get("status") in done_statuses or not RETRY_FAILED)]

# Ghi lại file checkpoint đã được làm sạch (loại bản ghi lỗi cần chạy lại, loại dòng hỏng, tránh trùng)
write_jsonl_atomic(JSONL_PATH, kept_records)
done_keys = {r["sample_key"] for r in kept_records}
pending_indices = [i for i in run_indices if sample_keys[i] not in done_keys]

print("Nguồn resume:", resume_source if resume_source else "(không có - chạy mới)")
print(f"Bản ghi hợp lệ đã có: {len(kept_records)} | dòng hỏng bỏ qua: {broken_line_count}")
print(f"Cần chạy: {len(pending_indices)} / {len(run_indices)} mẫu")

In [ ]:
# 6.3 Chạy inference có checkpoint; luôn ghi nốt buffer khi bị ngắt hoặc xảy ra lỗi
def fetch_image_data(row_index):
    # Đọc dữ liệu ảnh của một dòng; trả về (dữ liệu, thông báo lỗi)
    try:
        return pope_ds[int(row_index)]["image"], None
    except Exception as exc:
        return None, f"{type(exc).__name__}: {exc}"


def process_batch(batch_indices):
    # Chạy inference cho một nhóm chỉ số dòng và trả về danh sách bản ghi
    fetched = [fetch_image_data(i) for i in batch_indices]
    results = [None] * len(batch_indices)
    ok_positions = []
    for position, (image_data, fetch_error) in enumerate(fetched):
        if fetch_error:
            results[position] = make_result(None, None, STATUS_ERROR, f"image_fetch_error: {fetch_error}")
        else:
            ok_positions.append(position)
    if ok_positions:
        questions = [meta_records[batch_indices[p]]["question"] for p in ok_positions]
        images = [fetched[p][0] for p in ok_positions]
        for position, result in zip(ok_positions, run_inference_batch(questions, images)):
            results[position] = result
    return [build_record(i, result) for i, result in zip(batch_indices, results)]


buffer_records = []
status_counter = {STATUS_SUCCESS: 0, STATUS_INVALID: 0, STATUS_ERROR: 0}
start_time = time.time()
interrupted = False

progress = tqdm(total=len(pending_indices), desc="Inference")
try:
    for start in range(0, len(pending_indices), BATCH_SIZE):
        batch_indices = pending_indices[start:start + BATCH_SIZE]
        new_records = process_batch(batch_indices)
        buffer_records.extend(new_records)
        for record in new_records:
            status_counter[record["status"]] += 1
        progress.update(len(batch_indices))
        progress.set_postfix(status_counter)
        if len(buffer_records) >= SAVE_INTERVAL:
            append_jsonl(JSONL_PATH, buffer_records)  # Checkpoint
            buffer_records = []
            free_memory()
except KeyboardInterrupt:
    interrupted = True
    print("\nĐã dừng thủ công. Kết quả đã xử lý sẽ được lưu để có thể resume.")
finally:
    if buffer_records:
        append_jsonl(JSONL_PATH, buffer_records)
        buffer_records = []
    progress.close()
    free_memory()

elapsed = time.time() - start_time
print(f"\nHoàn tất lượt chạy trong {elapsed / 60:.1f} phút | {status_counter}")

---
# PHẦN 7: LƯU KẾT QUẢ
Tạo `vlm_predictions.csv`, `vlm_predictions.jsonl` (đã khử trùng, sắp theo thứ tự dataset) và `inference_summary.json`.

Quy ước cột `prediction`: `Yes` / `No` (hợp lệ), `Invalid` (model trả lời sai định dạng), rỗng (mẫu lỗi, `status = error`). Notebook evaluation chỉ tính metrics trên `status = success`.

In [ ]:
# 7.1 Tổng hợp kết quả cuối cùng và ghi CSV / JSONL
final_map, _ = read_jsonl_records(JSONL_PATH)
final_records = sorted((final_map[k] for k in run_keys if k in final_map), key=lambda r: r["row_index"])

if not final_records:
    raise RuntimeError("Chưa có kết quả nào để lưu.")

final_df = pd.DataFrame(final_records)
for column in OUTPUT_COLUMNS + EXTRA_COLUMNS:
    if column not in final_df.columns:
        final_df[column] = None
final_df = final_df[OUTPUT_COLUMNS + EXTRA_COLUMNS]

# Kiểm tra không trùng khoá
assert final_df["sample_key"].is_unique, "Phát hiện khoá mẫu trùng trong kết quả"

final_df.to_csv(CSV_PATH, index=False, encoding="utf-8")
write_jsonl_atomic(JSONL_PATH, final_records)  # Ghi từ bản ghi gốc (không có NaN) để JSON hợp lệ
print(f"Đã lưu {len(final_df)} dòng:\n - {CSV_PATH}\n - {JSONL_PATH}")

In [ ]:
# 7.2 Ghi inference_summary.json
n_success = int((final_df["status"] == STATUS_SUCCESS).sum())
n_invalid = int((final_df["status"] == STATUS_INVALID).sum())
n_error = int((final_df["status"] == STATUS_ERROR).sum())
n_pending = len(run_indices) - len(final_df)

# Số câu trả lời được parse thành công nhưng model có nói thêm chữ ngoài một từ Yes/No
parsed_rows = final_df[final_df["status"] == STATUS_SUCCESS]
extra_text_count = int(sum(
    re.sub(r"[\W_]+", "", str(raw).lower()) != str(pred).lower()
    for raw, pred in zip(parsed_rows["raw_output"], parsed_rows["prediction"])))

latency = pd.to_numeric(final_df["latency_ms"], errors="coerce").dropna()
latency_stats = ({"count": int(latency.size), "mean_ms": round(float(latency.mean()), 1),
                  "median_ms": round(float(latency.median()), 1), "p95_ms": round(float(latency.quantile(0.95)), 1)}
                 if latency.size else None)

is_complete_subset = bool(USE_SUBSET and MAX_SAMPLES is None and len(final_df) == 900 and n_pending == 0)
summary = {
    "model_name": MODEL_NAME,
    "model_source": MODEL_PATH or MODEL_NAME,
    "model_revision": MODEL_REVISION_RESOLVED,
    "dataset_name": DATASET_NAME,
    "dataset_revision": resolve_commit_sha(DATASET_NAME, repo_type="dataset"),
    "dataset_split": SPLIT_NAME,
    "dataset_files": PARQUET_FILES_USED,
    "data_source_mode": DATA_SOURCE,
    "subset_name": SUBSET_NAME if USE_SUBSET else None,
    "subset_ids_sha256": SUBSET_IDS_SHA256 if USE_SUBSET else None,
    "scope": "subset_900_not_full_9000" if USE_SUBSET else "full_split",
    "is_complete_subset": is_complete_subset,
    "total_samples_in_split": len(pope_ds),
    "total_samples": len(run_indices),
    "processed_samples": len(final_df),
    "pending_samples": n_pending,
    "successful_samples": n_success,
    "failed_samples": n_error,
    "invalid_predictions": n_invalid,
    "yes_count": int((final_df["prediction"] == "Yes").sum()),
    "no_count": int((final_df["prediction"] == "No").sum()),
    "parsed_but_extra_text_count": extra_text_count,
    "latency": latency_stats,
    "sample_key_mode": SAMPLE_KEY_MODE,
    "inference_timestamp_utc": dt.datetime.now(dt.timezone.utc).isoformat(),
    "inference_parameters": {
        "use_subset": USE_SUBSET, "subset_name": SUBSET_NAME, "max_samples": MAX_SAMPLES, "sampling": SAMPLING,
        "seed": SEED, "batch_size": BATCH_SIZE,
        "max_new_tokens": MAX_NEW_TOKENS, "do_sample": DO_SAMPLE,
        "temperature": TEMPERATURE if DO_SAMPLE else None,
        "max_image_side": MAX_IMAGE_SIDE, "oom_fallback_sides": list(OOM_FALLBACK_SIDES),
        "dtype": str(model.dtype), "attn_implementation": ATTN_IMPLEMENTATION, "trust_remote_code": True,
        "prompt_template": PROMPT_TEMPLATE, "phi4_chat_format": "<|user|><|image_1|>{prompt}<|end|><|assistant|>",
        "save_interval": SAVE_INTERVAL, "resume": RESUME, "retry_failed": RETRY_FAILED,
    },
    "device_allocation": DEVICE_ALLOCATION,
    "max_memory": {"gpu0": GPU0_MAX_MEMORY, "gpu1": GPU1_MAX_MEMORY, "single_gpu": SINGLE_GPU_MAX_MEMORY, "cpu": CPU_MAX_MEMORY},
    "environment": {
        "python": sys.version.split()[0], "torch": torch.__version__,
        "transformers": transformers.__version__, "accelerate": get_pkg_version("accelerate"),
        "huggingface_hub": get_pkg_version("huggingface_hub"),
        "gpu": [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())] if torch.cuda.is_available() else None,
        "cuda": torch.version.cuda,
    },
    "interrupted": interrupted,
}
with open(SUMMARY_PATH, "w", encoding="utf-8") as file:
    json.dump(summary, file, ensure_ascii=False, indent=2, default=json_default)
print(json.dumps(summary, ensure_ascii=False, indent=2, default=json_default))
if n_pending > 0:
    print(f"\nCòn {n_pending} mẫu chưa xử lý. Chạy lại cell 6.2 -> 6.3 -> 7.1 -> 7.2 để tiếp tục (resume).")


In [ ]:
# 7.3 Kết quả cuối: 10 mẫu đầu, thống kê Yes/No, mẫu lỗi, đường dẫn file
try:
    display(final_df.head(10))
except NameError:
    print(final_df.head(10).to_string())

print("\nThống kê prediction:")
print(final_df["prediction"].value_counts(dropna=False))
print(f"\nYes: {summary['yes_count']} | No: {summary['no_count']} | Invalid: {n_invalid} | Lỗi (error): {n_error}")

if n_error:
    print("\nMột số mẫu lỗi:")
    print(final_df[final_df["status"] == STATUS_ERROR][["id", "question", "error"]].head(10).to_string())

print("\nFile kết quả:")
for output_path in [CSV_PATH, JSONL_PATH, SUMMARY_PATH]:
    print(f" - {output_path.resolve()}  ({output_path.stat().st_size / 1024:.1f} KB)")

In [ ]:
# 7.4 Kiểm tra cuối: đủ tập con, 300 mẫu/category, thống kê theo category, đóng gói để tải về
import shutil

check_df = final_df.copy()
print("Số dòng kết quả:", len(check_df), "| khoá mẫu duy nhất:", check_df["sample_key"].is_unique)

if USE_SUBSET and MAX_SAMPLES is None:
    expected_keys = {k for ids in SUBSET_IDS_BY_CATEGORY.values() for k in ids}
    got_keys = set(check_df["sample_key"].astype(str))
    print("Đủ 900 mẫu của tập con:", got_keys == expected_keys,
          "| thiếu:", len(expected_keys - got_keys), "| thừa:", len(got_keys - expected_keys))
    counts = check_df["category"].value_counts().to_dict()
    print("Số mẫu theo category:", counts)
    assert got_keys == expected_keys and all(v == 300 for v in counts.values()), \
        "Kết quả chưa khớp tập con 900 mẫu (300/category) — chưa được dùng làm kết quả cuối"
else:
    print("CẢNH BÁO: đang ở chế độ chạy thử/toàn split (USE_SUBSET=False hoặc MAX_SAMPLES != None) -> không phải kết quả cuối.")

table = (check_df.assign(is_yes=(check_df["prediction"] == "Yes"),
                         is_success=(check_df["status"] == STATUS_SUCCESS),
                         is_invalid=(check_df["status"] == STATUS_INVALID),
                         is_error=(check_df["status"] == STATUS_ERROR))
         .groupby("category")[["is_success", "is_invalid", "is_error", "is_yes"]].sum())
table["n"] = check_df.groupby("category").size()
table["yes_ratio"] = (table["is_yes"] / table["n"]).round(3)
print("\nTổng hợp theo category (chưa phải metric; chấm điểm ở notebook evaluation):")
print(table.to_string())

archive = shutil.make_archive(str(WORK_DIR / "phi4_pope_subset900_output"), "zip", root_dir=OUTPUT_DIR)
print("\nĐã tạo file để tải về:", archive)
